# 03 Granite explanations

The decision takes milliseconds. After that Granite (gguf, llama.cpp, runs on this LinuxONE server, nothing is sent outside) writes a short message for the customer and a summary for the analyst. Same text goes into the email to the trusted contact.

In [1]:
import os, sys, time, json, platform
from pathlib import Path
ROOT = Path.cwd().parent if (Path.cwd().parent / "safepause").exists() else Path.cwd()
sys.path.insert(0, str(ROOT))
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
from safepause import config
print("project:", ROOT, "| machine:", platform.machine(), "| CPUs:", os.cpu_count())

project: /home/ibm-user/shared/safepause | machine: s390x | CPUs: 3


In [2]:
from safepause.scorer import Scorer
from safepause.features import gather_sequences
from safepause.genai import Explainer, find_granite_model, facts_text, template_customer
from safepause.alerts import EmailAlerter
import glob, importlib.metadata
path = find_granite_model()
print("Granite model:", path, f"({os.path.getsize(path)/1e6:.0f} MB)" if path else "")
try: print("llama_cpp_python:", importlib.metadata.version("llama_cpp_python"))
except Exception: print("llama_cpp_python not installed")
print("zDNN llama backend:", glob.glob("/opt/venv/lib*/python3*/site-packages/llama_cpp/lib/libggml-zdnn*.so") or "not found")

Granite model: /home/ibm-user/granite-4.0-h-350m-base-be.f16.gguf (685 MB)
llama_cpp_python: 0.3.36
zDNN llama backend: ['/opt/venv/lib/python3.12/site-packages/llama_cpp/lib/libggml-zdnn.so', '/opt/venv/lib64/python3.12/site-packages/llama_cpp/lib/libggml-zdnn.so']


In [3]:
d = np.load(config.DATASET_NPZ); X, y, prev_idx, amount = d["X"], d["y"], d["prev_idx"], d["amount"]
te = np.arange(int(d["test_start"]), len(X))
stream = pd.read_csv(config.STREAM_CSV, dtype={"mcc": str, "use_chip": str})
scorer = Scorer()
risk, dec, px, ps, rules = scorer.score_batch(X[te], gather_sequences(X, prev_idx, te))
paused = np.flatnonzero((dec == "pause") & (y[te] == 1))
if len(paused) == 0:
    paused = np.argsort(-risk)[:3]
print(f"{(dec=='pause').sum()} paused, {(dec=='review').sum()} review in test period; picking 3 real fraud cases")
cases = []
for k in paused[np.argsort(-risk[paused])][[0, len(paused)//2, -1]] if len(paused) >= 3 else paused:
    i = te[k]
    cases.append({"tx_id": str(stream.tx_id.iloc[k]), "amount": float(amount[i]), "mcc": str(stream.mcc.iloc[k]),
                  "use_chip": str(stream.use_chip.iloc[k]), "risk": float(risk[k]), "decision": "pause",
                  "reasons": scorer.reasons(X[i])})
print(facts_text(cases[0]))

22 paused, 1027 review in test period; picking 3 real fraud cases
Payment amount: $85.14
Merchant category code: 5621
Payment type: in person
Risk score: 0.78 (0 = safe, 1 = almost certainly fraud)
Decision: pause
Warning signals:
- Spent 34x the usual amount within 24 hours
- 4 new recipients within 24 hours
- Recipient never paid before


In [4]:
explainer = Explainer()
t0 = time.perf_counter(); explainer.load(); print(f"model load {time.perf_counter()-t0:.1f}s")
for c in cases:
    cm, src, ms = explainer.customer_message(c)
    am, _, ms2 = explainer.analyst_summary(c)
    c.update(customer_msg=cm, analyst_msg=am)
    print("=" * 100)
    print(f"case {c['tx_id']}  ${abs(c['amount']):,.2f}  risk {c['risk']:.2f}  reasons: {c['reasons']}")
    print(f"\nCUSTOMER ({src}, {ms/1000:.1f}s):\n{cm}")
    print(f"\nANALYST ({ms2/1000:.1f}s):\n{am}")
    print(f"\nTEMPLATE fallback for comparison:\n{template_customer(c)}")

llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 256


[genai] Granite loaded from /home/ibm-user/granite-4.0-h-350m-base-be.f16.gguf in 0.8s (base model, few-shot completion)
model load 0.8s


case 209376  $85.14  risk 0.78  reasons: ['Spent 34x the usual amount within 24 hours', '4 new recipients within 24 hours', 'Recipient never paid before']

CUSTOMER (granite, 11.8s):
We paused your payment of $85.14 because there were several payments within 24 hours and 4 new recipients within 24 hours. If anyone is asking you to pay or is using your card, please stop and contact us right away. If it was you, simply confirm the payment.

ANALYST (7.4s):
ELEVATED risk (0.78). Spent 34x the usual amount within 24 hours, with 4 new recipients within 24 hours. Recipient never paid before. Pattern matches an authorised push payment scam. Action: keep on hold and call the customer before release.

TEMPLATE fallback for comparison:
We paused your payment of $85.14 to protect you: spent 34x the usual amount within 24 hours; 4 new recipients within 24 hours. If someone called or messaged you and asked you to send money urgently, please stop and contact us. If it was really you, just confirm th

case 209380  $33.40  risk 0.55  reasons: ['Spent 35x the usual amount within 24 hours', '5 new recipients within 24 hours', 'Recipient never paid before']

CUSTOMER (granite, 10.8s):
We paused your payment of $33.40 because there were several payments within 24 hours and 5 new recipients within 24 hours. If anyone is asking you to pay or is using your card, please stop and contact us right away. If it was you, simply confirm the payment.

ANALYST (7.7s):
ELEVATED risk (0.55). Spent 35x the usual amount within 24 hours, with 5 new recipients within 24 hours. Recipient never paid before. Pattern matches an authorised push payment scam. Action: keep on hold and call the customer before release.

TEMPLATE fallback for comparison:
We paused your payment of $33.40 to protect you: spent 35x the usual amount within 24 hours; 5 new recipients within 24 hours. If someone called or messaged you and asked you to send money urgently, please stop and contact us. If it was really you, just confirm th

case 209382  $85.00  risk 0.44  reasons: ['4 payments in the previous hour', 'Spent 37x the usual amount within 24 hours', '5 new recipients within 24 hours']

CUSTOMER (granite, 11.1s):
We paused your payment of $85.00 because there were several payments within an hour and a new recipient was added within 24 hours. If anyone is asking you to pay or is using your card, please stop and contact us right away. If it was you, simply confirm the payment.

ANALYST (7.5s):
ELEVATED risk (0.44). 4 payments in the previous hour, spent 37x the usual amount within 24 hours, and 5 new recipients within 24 hours. Pattern matches an authorised push payment scam. Action: keep on hold and call the customer before release.

TEMPLATE fallback for comparison:
We paused your payment of $85.00 to protect you: 4 payments in the previous hour; spent 37x the usual amount within 24 hours. If someone called or messaged you and asked you to send money urgently, please stop and contact us. If it was really you, j

## Email

In [5]:
msg = EmailAlerter().build(cases[0], cases[0]["customer_msg"], cases[0]["analyst_msg"])
print("Subject:", msg["Subject"]); print(); print(msg.get_content())

Subject: SafePause: a $85.14 payment by your family member was paused

Hello,

You are the trusted contact for a bank customer. SafePause has just paused a payment of $85.14 because it looks like a possible scam.

Why it was paused:
  - Spent 34x the usual amount within 24 hours
  - 4 new recipients within 24 hours
  - Recipient never paid before

What the customer sees:
  "We paused your payment of $85.14 because there were several payments within 24 hours and 4 new recipients within 24 hours. If anyone is asking you to pay or is using your card, please stop and contact us right away. If it was you, simply confirm the payment."

Please call them now and check that nobody is pressuring them to send money. Scammers often pretend to be the bank, the police or a relative in trouble.

Case reference: 209376 (risk 0.78)

This alert contains no account numbers or personal identifiers.
SafePause

